In [30]:
from pathlib import Path
import sys

filepath = Path("../data/graph_output.json")

In [31]:
# json load
import json
loaded_data = json.loads(filepath.read_text(encoding="utf-8"))

In [32]:
# create a mapping from URI to the person nodes that share it
class UriPersonMap(dict):
    def append(self, item):
        if not isinstance(item, dict):
            raise TypeError("item must be a dict with a single URI -> person entry")

        uri, person = next(iter(item.items()))
        self.setdefault(uri, []).append(person)

uri_person_map = UriPersonMap()

for node in loaded_data.get("nodes", []):
    if node.get("data", {}).get("label") == "PERSON":
        person = node["data"]
        uri = person.get("uri")
        if uri:
            uri_person_map.append({uri: person})

print(f"Found {len(uri_person_map)} unique URI groups")


Found 252 unique URI groups


In [33]:
out_path = filepath.parent / "uri_person_map.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(dict(uri_person_map), f, ensure_ascii=False, indent=2)
print(f"Saved {len(uri_person_map)} groups to {out_path}")

Saved 252 groups to ../data/uri_person_map.json


In [ ]:
import json
from pathlib import Path
from collections import defaultdict

ROOT_DIR = Path("..")
INPUT_PATH = ROOT_DIR / "data" / "uri_person_map.json"
OUTPUT_PATH = ROOT_DIR / "data" / "graph_output.json"


def build_elements(input_path: Path) -> dict:
    with input_path.open("r", encoding="utf-8") as handle:
        grouped_people = json.load(handle)

    nodes = []
    edges = []

    person_node_ids = {}
    article_node_ids = {}
    article_person_names = defaultdict(set)

    node_id = 1
    edge_id = 100000

    # First pass: collect article -> person names and create person nodes.
    for uri, people in grouped_people.items():
        names = []
        for person in people:
            name = (person.get("name") or "").strip()
            if name:
                names.append(name)
                article = (person.get("article") or "").strip()
                if article:
                    article_person_names[article].add(name)

        primary_name = names[0] if names else uri
        person_node_id = node_id
        node_id += 1

        person_node_ids[uri] = person_node_id
        nodes.append(
            {
                "data": {
                    "id": person_node_id,
                    "label": "PERSON",
                    "name": primary_name,
                    "names": names,
                    "alternate_name": [name for name in names if name != primary_name],
                    "uri": uri,
                }
            }
        )

    # Second pass: create article nodes.
    for article, person_names in sorted(article_person_names.items()):
        article_node_id = node_id
        node_id += 1
        article_node_ids[article] = article_node_id
        nodes.append(
            {
                "data": {
                    "id": article_node_id,
                    "label": "ARTICLE",
                    "name": article,
                    "person_names": sorted(person_names),
                }
            }
        )

    # Third pass: connect person nodes to article nodes.
    for uri, people in grouped_people.items():
        person_node_id = person_node_ids[uri]
        article_names = {
            (person.get("article") or "").strip()
            for person in people
            if (person.get("article") or "").strip()
        }
        for article in sorted(article_names):
            if article in article_node_ids:
                edges.append(
                    {
                        "data": {
                            "id": edge_id,
                            "label": "MENTIONS_ARTICLE",
                            "source": person_node_id,
                            "target": article_node_ids[article],
                        }
                    }
                )
                edge_id += 1

    return {"nodes": nodes, "edges": edges}


if __name__ == "__main__":
    elements = build_elements(INPUT_PATH)
    with OUTPUT_PATH.open("w", encoding="utf-8") as handle:
        json.dump(elements, handle, ensure_ascii=False, indent=2)
    print(f"Wrote {len(elements['nodes'])} nodes and {len(elements['edges'])} edges to {OUTPUT_PATH}")


Wrote 375 nodes and 314 edges to ../data/graph_output.json
